# Capítulo 9. Evaluación y comparación de modelos

**Aprendizaje y Clasificación Automática con R**  
**Autor:** Jesús Gilberto Rodríguez Escobedo

Este cuaderno es **independiente y autónomo**: puede abrirse directamente sin ejecutar capítulos anteriores.

1. Ejecute primero la celda **Preparación automática y autónoma del capítulo**.
2. Después ejecute las celdas en orden.
3. Si Colab reinicia la sesión, vuelva a ejecutar desde la primera celda.

[Volver al índice de cuadernos Colab](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/00-indice-colabs.ipynb)


In [ ]:
# Preparación automática y autónoma del capítulo
options(repos = c(CRAN = "https://cloud.r-project.org"))

paquetes_libro <- c(
  "ggplot2", "readr", "dplyr", "tidyr", "stringr", "data.table",
  "class", "rpart", "randomForest", "ranger", "e1071", "naivebayes",
  "neuralnet", "cluster", "caret", "factoextra", "scales", "plotly", "DT"
)
faltantes <- paquetes_libro[!vapply(paquetes_libro, requireNamespace, logical(1), quietly = TRUE)]
if (length(faltantes)) install.packages(faltantes)

dir.create("datos/covid19/procesados", showWarnings = FALSE, recursive = TRUE)
dir.create("datos/covid19/muestras", showWarnings = FALSE, recursive = TRUE)
dir.create("datos/covid19/diccionarios", showWarnings = FALSE, recursive = TRUE)

archivos_colab <- c(
  "util_graficas.R" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/util_graficas.R",
  "datos/atus_ml_preparado.csv" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/atus_ml_preparado.csv",
  "datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz",
  "datos/covid19/muestras/covid19_mexico_2022_muestra.csv.gz" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/covid19/muestras/covid19_mexico_2022_muestra.csv.gz",
  "datos/covid19/diccionarios/diccionario_covid19_ml.csv" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/covid19/diccionarios/diccionario_covid19_ml.csv"
)
for (destino in names(archivos_colab)) {
  if (!file.exists(destino)) download.file(archivos_colab[[destino]], destino, mode = "wb", quiet = TRUE)
}
stopifnot(all(file.exists(names(archivos_colab))))
source("util_graficas.R")
cat("Entorno autónomo listo. R:", R.version.string, "\n")


# Evaluación, validación y comparación de modelos

La formulación matemática de **matrices de confusión, métricas, validación y selección de modelos** se desarrolla con mayor profundidad
en los capítulos 6 y 7 de *Fundamentos Matemáticos del Aprendizaje
Automático* [@rodriguez2026fundamentos].

## Objetivos del capítulo

Al finalizar este capítulo, el lector será capaz de:

- explicar por qué la exactitud no siempre es suficiente;
- construir e interpretar una matriz de confusión;
- calcular exactitud, sensibilidad, especificidad, precisión y valor F1;
- distinguir entrenamiento, validación y prueba;
- aplicar validación cruzada estratificada;
- comparar modelos con criterios predictivos y prácticos;
- seleccionar un modelo sin depender de una sola métrica.

## Introducción

Entrenar un modelo es apenas la mitad del trabajo. Después debemos responder una pregunta más delicada: **¿qué tan bien funcionará con accidentes que nunca vio durante el aprendizaje?**

Un modelo puede memorizar los datos de entrenamiento y parecer excelente, pero fallar con observaciones nuevas. Por eso la evaluación debe realizarse con datos separados y métricas apropiadas para el objetivo del problema.

En este capítulo se utiliza nuevamente la base preparada a partir de ATUS [@inegi_atus_2024].

## Cargar los datos


In [ ]:
library(readr)
library(dplyr)
library(ggplot2)
source("util_graficas.R")

ruta_atus_ml <- "datos/atus_ml_preparado.csv"

if (!file.exists(ruta_atus_ml)) {
  stop(
    paste(
      "No se encontró datos/atus_ml_preparado.csv.",
      "Renderice primero el capítulo de preparación de datos."
    )
  )
}

atus_ml <- read_csv(
  ruta_atus_ml,
  show_col_types = FALSE
)


## Preparar una muestra reproducible


In [ ]:
set.seed(123)

atus_eval <- atus_ml |>
  sample_n(min(30000, nrow(atus_ml))) |>
  mutate(
    accidente_con_victimas = factor(
      accidente_con_victimas,
      levels = c("Solo daños", "Con víctimas")
    ),
    MES = factor(MES),
    ID_HORA = as.numeric(ID_HORA),
    DIASEMANA = factor(DIASEMANA),
    TIPACCID = factor(TIPACCID),
    CAUSAACCI = factor(CAUSAACCI)
  ) |>
  na.omit()

prop.table(table(atus_eval$accidente_con_victimas))


La tabla de proporciones permite identificar si una clase es mucho más frecuente que la otra. Cuando existe desbalance, una exactitud alta puede resultar engañosa.

## Entrenamiento, validación y prueba

Los datos pueden dividirse en tres partes:

- **Entrenamiento:** se utiliza para estimar los parámetros del modelo.
- **Validación:** ayuda a elegir hiperparámetros y comparar alternativas.
- **Prueba:** se reserva para estimar el desempeño final.

En ejercicios introductorios es frecuente utilizar entrenamiento y prueba, mientras que la validación se realiza mediante validación cruzada dentro del conjunto de entrenamiento.

## División estratificada

Una división estratificada conserva aproximadamente la proporción de cada clase.


In [ ]:
set.seed(123)

indices_entrenamiento <- unlist(
  lapply(
    split(seq_len(nrow(atus_eval)), atus_eval$accidente_con_victimas),
    function(indices) {
      sample(indices, size = floor(0.70 * length(indices)))
    }
  )
)

entrenamiento <- atus_eval[indices_entrenamiento, ]
prueba <- atus_eval[-indices_entrenamiento, ]

prop.table(table(entrenamiento$accidente_con_victimas))
prop.table(table(prueba$accidente_con_victimas))


Los índices se separan por clase y después se toma 70 % de cada grupo. Así se reduce el riesgo de que una clase quede sobrerrepresentada en entrenamiento o prueba.

## Matriz de confusión

Para una clase positiva denominada **Con víctimas**, la matriz contiene:

- **Verdadero positivo (VP):** el accidente tenía víctimas y el modelo lo detectó.
- **Falso negativo (FN):** tenía víctimas, pero el modelo lo clasificó como solo daños.
- **Falso positivo (FP):** era de solo daños, pero el modelo predijo víctimas.
- **Verdadero negativo (VN):** era de solo daños y se clasificó correctamente.

| Clase real | Predicción positiva | Predicción negativa |
|---|---:|---:|
| Positiva | VP | FN |
| Negativa | FP | VN |

Un falso negativo puede ser especialmente importante cuando el propósito consiste en identificar accidentes con víctimas.

## Métricas principales

### Exactitud

$$
\text{Exactitud}=\frac{VP+VN}{VP+FN+FP+VN}
$$

Mide la proporción total de predicciones correctas.

### Sensibilidad

$$
\text{Sensibilidad}=\frac{VP}{VP+FN}
$$

Mide qué proporción de los accidentes con víctimas fue detectada.

### Especificidad

$$
\text{Especificidad}=\frac{VN}{VN+FP}
$$

Mide qué proporción de los accidentes de solo daños fue reconocida correctamente.

### Precisión

$$
\text{Precisión}=\frac{VP}{VP+FP}
$$

Indica qué proporción de las predicciones positivas realmente tenía víctimas.

### Valor F1

$$
F_1=2\frac{\text{Precisión}\times\text{Sensibilidad}}
{\text{Precisión}+\text{Sensibilidad}}
$$

El valor F1 equilibra precisión y sensibilidad.

## Función para calcular métricas


In [ ]:
calcular_metricas <- function(real, predicho, positiva = "Con víctimas") {
  real <- factor(real)
  predicho <- factor(predicho, levels = levels(real))

  negativas <- setdiff(levels(real), positiva)

  if (length(negativas) != 1) {
    stop("La función requiere exactamente dos clases.")
  }

  negativa <- negativas[1]
  matriz <- table(Real = real, Predicho = predicho)

  VP <- matriz[positiva, positiva]
  FN <- matriz[positiva, negativa]
  FP <- matriz[negativa, positiva]
  VN <- matriz[negativa, negativa]

  division_segura <- function(numerador, denominador) {
    if (
      is.na(numerador) ||
      is.na(denominador) ||
      denominador == 0
    ) {
      return(NA_real_)
    }

    as.numeric(numerador / denominador)
  }

  exactitud <- division_segura(VP + VN, VP + FN + FP + VN)
  sensibilidad <- division_segura(VP, VP + FN)
  especificidad <- division_segura(VN, VN + FP)
  precision <- division_segura(VP, VP + FP)
  f1 <- division_segura(
    2 * precision * sensibilidad,
    precision + sensibilidad
  )

  list(
    matriz = matriz,
    metricas = data.frame(
      exactitud = exactitud,
      sensibilidad = sensibilidad,
      especificidad = especificidad,
      precision = precision,
      f1 = f1
    )
  )
}


## Modelo de referencia: regla mayoritaria

Antes de celebrar un modelo complejo, conviene compararlo con una regla muy simple: predecir siempre la clase más frecuente.


In [ ]:
clase_mayoritaria <- names(
  which.max(table(entrenamiento$accidente_con_victimas))
)

prediccion_referencia <- factor(
  rep(clase_mayoritaria, nrow(prueba)),
  levels = levels(entrenamiento$accidente_con_victimas)
)

resultado_referencia <- calcular_metricas(
  real = prueba$accidente_con_victimas,
  predicho = prediccion_referencia
)

resultado_referencia$matriz
resultado_referencia$metricas


El modelo de referencia predice únicamente la clase mayoritaria. Si nunca predice la clase positiva **Con víctimas**, la precisión no puede calcularse porque su denominador es cero. En ese caso R muestra `NA`, que significa **métrica no definida**, pero el libro continúa procesándose normalmente.

Si 90 % de los accidentes perteneciera a una sola clase, predecir siempre esa clase produciría 90 % de exactitud sin aprender ninguna relación útil. Por eso deben revisarse sensibilidad, especificidad, precisión y F1.

## Evaluar una regresión logística


In [ ]:
modelo_logistico <- glm(
  accidente_con_victimas ~
    MES + ID_HORA + DIASEMANA + TIPACCID + CAUSAACCI,
  data = entrenamiento,
  family = binomial
)

probabilidad_logistica <- predict(
  modelo_logistico,
  newdata = prueba,
  type = "response"
)

# glm() modela la probabilidad del segundo nivel del factor.
clase_positiva_modelada <- levels(
  entrenamiento$accidente_con_victimas
)[2]

prediccion_logistica <- ifelse(
  probabilidad_logistica >= 0.50,
  clase_positiva_modelada,
  levels(entrenamiento$accidente_con_victimas)[1]
)

prediccion_logistica <- factor(
  prediccion_logistica,
  levels = levels(entrenamiento$accidente_con_victimas)
)

resultado_logistico <- calcular_metricas(
  real = prueba$accidente_con_victimas,
  predicho = prediccion_logistica
)

resultado_logistico$matriz
resultado_logistico$metricas


En una regresión logística binaria, `glm()` calcula la probabilidad del segundo nivel del factor respuesta. El orden de los niveles debe verificarse antes de transformar probabilidades en clases.

## Comparación inicial


In [ ]:
comparacion_inicial <- bind_rows(
  cbind(
    modelo = "Regla mayoritaria",
    resultado_referencia$metricas
  ),
  cbind(
    modelo = "Regresión logística",
    resultado_logistico$metricas
  )
)

comparacion_inicial


In [ ]:
comparacion_larga <- comparacion_inicial |>
  tidyr::pivot_longer(
    cols = -modelo,
    names_to = "metrica",
    values_to = "valor"
  )

ggplot(
  comparacion_larga,
  aes(x = metrica, y = valor, fill = modelo)
) +
  geom_col(position = "dodge") +
  scale_y_continuous(
    limits = c(0, 1),
    labels = scales::label_percent(accuracy = 1)
  ) +
  labs(
    title = "Un modelo debe superar una referencia sencilla",
    x = "Métrica",
    y = "Valor",
    fill = "Modelo"
  ) +
  tema_libro()


**Fuente:** Elaboración propia con datos del INEGI, ATUS 2024.

## Umbral de clasificación

El umbral de 0.50 no es una ley universal. Reducirlo puede aumentar la sensibilidad, aunque también puede generar más falsos positivos.


In [ ]:
umbrales <- seq(0.10, 0.90, by = 0.05)

metricas_umbral <- lapply(umbrales, function(umbral) {
  prediccion <- ifelse(
    probabilidad_logistica >= umbral,
    clase_positiva_modelada,
    levels(entrenamiento$accidente_con_victimas)[1]
  )

  prediccion <- factor(
    prediccion,
    levels = levels(entrenamiento$accidente_con_victimas)
  )

  metricas <- calcular_metricas(
    prueba$accidente_con_victimas,
    prediccion
  )$metricas

  cbind(umbral = umbral, metricas)
}) |>
  bind_rows()

metricas_umbral


El mejor umbral depende del costo de cada error. Cuando omitir un accidente con víctimas es más grave que generar una alerta adicional, puede priorizarse la sensibilidad.

## Validación cruzada

La validación cruzada divide el conjunto de entrenamiento en varios pliegues. Cada pliegue funciona una vez como validación y los restantes como entrenamiento.

En validación cruzada de $K$ pliegues:

1. se divide la muestra en $K$ partes;
2. se entrena con $K-1$ partes;
3. se evalúa en la parte restante;
4. se repite hasta utilizar todos los pliegues;
5. se promedian las métricas.

### Crear pliegues estratificados


In [ ]:
crear_pliegues <- function(respuesta, k = 5, semilla = 123) {
  set.seed(semilla)

  pliegues <- integer(length(respuesta))

  for (clase in levels(factor(respuesta))) {
    indices <- which(respuesta == clase)
    etiquetas <- rep(seq_len(k), length.out = length(indices))
    pliegues[indices] <- sample(etiquetas)
  }

  pliegues
}

pliegue <- crear_pliegues(
  entrenamiento$accidente_con_victimas,
  k = 5
)

table(pliegue, entrenamiento$accidente_con_victimas)


### Validación cruzada de la regresión logística


In [ ]:
resultados_cv <- lapply(1:5, function(k) {
  datos_entrenamiento <- entrenamiento[pliegue != k, ]
  datos_validacion <- entrenamiento[pliegue == k, ]

  modelo <- glm(
    accidente_con_victimas ~
      MES + ID_HORA + DIASEMANA + TIPACCID + CAUSAACCI,
    data = datos_entrenamiento,
    family = binomial
  )

  probabilidad <- predict(
    modelo,
    newdata = datos_validacion,
    type = "response"
  )

  clase_modelada <- levels(
    datos_entrenamiento$accidente_con_victimas
  )[2]

  prediccion <- ifelse(
    probabilidad >= 0.50,
    clase_modelada,
    levels(datos_entrenamiento$accidente_con_victimas)[1]
  )

  prediccion <- factor(
    prediccion,
    levels = levels(datos_entrenamiento$accidente_con_victimas)
  )

  metricas <- calcular_metricas(
    datos_validacion$accidente_con_victimas,
    prediccion
  )$metricas

  cbind(pliegue = k, metricas)
}) |>
  bind_rows()

resultados_cv


In [ ]:
resumen_cv <- resultados_cv |>
  summarise(
    across(
      where(is.numeric) & !matches("pliegue"),
      list(
        media = ~ mean(.x, na.rm = TRUE),
        desviacion = ~ sd(.x, na.rm = TRUE)
      )
    )
  )

resumen_cv


La media resume el desempeño esperado y la desviación estándar muestra su estabilidad. Dos modelos con medias parecidas pueden diferir mucho si uno presenta resultados más variables entre pliegues.

## Cómo comparar varios modelos

Para comparar regresión logística, k-NN, árbol de decisión y Random Forest conviene utilizar:

1. la misma muestra;
2. la misma variable respuesta;
3. la misma partición de entrenamiento y prueba;
4. el mismo criterio para definir la clase positiva;
5. las mismas métricas;
6. una validación cruzada equivalente;
7. tiempos de entrenamiento y predicción;
8. facilidad de interpretación.

Una tabla de decisión puede tener esta estructura:

| Modelo | Exactitud | Sensibilidad | Especificidad | F1 | Interpretación | Costo computacional |
|---|---:|---:|---:|---:|---|---|
| Regresión logística |  |  |  |  | Alta | Bajo |
| k-NN |  |  |  |  | Media-baja | Medio |
| Árbol de decisión |  |  |  |  | Alta | Bajo |
| Random Forest |  |  |  |  | Media | Alto |

## Selección del modelo final

No existe un modelo universalmente mejor. La selección depende de la finalidad:

- Si se requiere explicar el efecto de las variables, puede preferirse regresión logística.
- Si se necesita una regla visual y fácil de comunicar, un árbol puede ser apropiado.
- Si se prioriza capacidad predictiva y estabilidad, Random Forest puede resultar competitivo.
- Si el costo de los falsos negativos es alto, debe priorizarse sensibilidad.
- Si las alertas falsas son costosas, precisión y especificidad cobran mayor importancia.

El mejor modelo no es necesariamente el que tiene la mayor exactitud, sino el que responde mejor al objetivo, controla los errores relevantes y mantiene un desempeño estable con datos nuevos.

## Actividad guiada

1. Ejecute la regresión logística con umbrales de 0.30, 0.50 y 0.70.
2. Registre la matriz de confusión de cada caso.
3. Compare sensibilidad, especificidad, precisión y F1.
4. Explique qué umbral elegiría para detectar accidentes con víctimas.
5. Justifique su decisión considerando el costo de falsos negativos y falsos positivos.

## Actividades para el lector

1. Construya una función que calcule la exactitud balanceada:

   $$
   \text{Exactitud balanceada}=\frac{\text{Sensibilidad}+\text{Especificidad}}{2}
   $$

2. Compare la variabilidad de las métricas en validación cruzada.
3. Repita el procedimiento con un árbol de decisión.
4. Elabore una tabla comparativa con los cuatro modelos estudiados.
5. Escriba una recomendación técnica de no más de 200 palabras.

## Resumen del capítulo

En este capítulo aprendimos que evaluar un modelo requiere datos no utilizados durante el entrenamiento, una clase positiva bien definida y varias métricas. También construimos una regla de referencia, estudiamos el efecto del umbral y aplicamos validación cruzada estratificada. Estas herramientas permiten comparar modelos de forma más justa y elegirlos según el objetivo real del análisis.

## Referencias fundamentales sobre evaluación

La validación cruzada y su uso para estimar el desempeño de modelos fueron examinados comparativamente por @kohavi1995crossvalidation. Para el análisis mediante curvas ROC y AUC, una referencia ampliamente utilizada es @fawcett2006roc. Una visión integrada de partición de datos, remuestreo y comparación de modelos aparece en @james2021islr.

## Laboratorio interactivo: matriz de confusión y métricas

Modifica los cuatro valores de la matriz de confusión y observa cómo cambian
exactitud, sensibilidad, especificidad, precisión y F1.


**Laboratorio interactivo:** este bloque se ejecuta en la versión web mediante Shinylive; aquí se conserva el desarrollo reproducible del capítulo.


### Laboratorio disponible en la versión web

La calculadora permite editar la matriz de confusión y obtener automáticamente
exactitud, sensibilidad, especificidad, precisión y F1.

## Materiales complementarios del capítulo

Estos recursos permiten repasar los conceptos principales del capítulo mediante distintos formatos. La presentación puede consultarse en PDF o modificarse en PowerPoint; la infografía ofrece una síntesis visual, el video explica los contenidos y el cuaderno Colab permite ejecutar los ejemplos de manera autónoma.

| Recurso | Utilidad | Abrir o reproducir | Descargar |
|---|---|---|---|
| Video explicativo | Explicación audiovisual de los contenidos del capítulo. | [Ver en YouTube](https://www.youtube.com/watch?v=ShiKrfhIfpY) | — |
| Presentación en PDF | Diapositivas para lectura, estudio o exposición. | [Ver PDF](recursos/capitulo-09/capitulo-09-evaluacion-modelos-presentacion.pdf) | [Descargar PDF](recursos/capitulo-09/capitulo-09-evaluacion-modelos-presentacion.pdf){download="capitulo-09-evaluacion-modelos-presentacion.pdf"} |
| Presentación editable | Archivo PowerPoint para utilizarlo en clase o adaptarlo. | [Abrir PPTX](recursos/capitulo-09/capitulo-09-evaluacion-modelos-presentacion.pptx) | [Descargar PPTX](recursos/capitulo-09/capitulo-09-evaluacion-modelos-presentacion.pptx){download="capitulo-09-evaluacion-modelos-presentacion.pptx"} |
| Infografía | Resumen visual de las ideas fundamentales. | [Ver infografía](recursos/capitulo-09/capitulo-09-evaluacion-modelos-infografia.png) | [Descargar PNG](recursos/capitulo-09/capitulo-09-evaluacion-modelos-infografia.png){download="capitulo-09-evaluacion-modelos-infografia.png"} |
| Cuaderno Google Colab | Cuaderno autónomo para ejecutar los ejemplos del capítulo sin necesidad de ejecutar los capítulos anteriores. | [Abrir en Google Colab](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/09-evaluacion-modelos.ipynb) | — |

### Video explicativo

### Vista previa de la infografía

[![Infografía del capítulo 9](recursos/capitulo-09/capitulo-09-evaluacion-modelos-infografia.png)](recursos/capitulo-09/capitulo-09-evaluacion-modelos-infografia.png)

**Video del capítulo:** <https://www.youtube.com/watch?v=ShiKrfhIfpY>

La presentación PDF, el archivo editable, la infografía y el cuaderno Colab pueden consultarse desde la versión web del libro.

Los materiales complementarios fueron elaborados con apoyo de **NotebookLM de Google**, a partir del contenido del capítulo, y posteriormente revisados y adaptados por el autor. El texto del libro y sus archivos fuente constituyen la referencia principal.

Este video forma parte de la lista oficial del curso **Aprendizaje y Clasificación Automática con R**.

[Consultar todos los videos del curso](https://www.youtube.com/playlist?list=PLDJYd2v7Kt-Q)

## Caso aplicado B: evaluación del modelo COVID-19

La evaluación debe realizarse con observaciones que no participaron en el
ajuste del modelo.


In [ ]:
library(readr)
library(dplyr)
library(tidyr)

covid <- read_csv(
  "datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz",
  show_col_types = FALSE
)

covid_modelo <- covid |>
  select(
    MURIO,
    EDAD,
    NEUMONIA,
    DIABETES,
    HIPERTENSION,
    OBESIDAD,
    RENAL_CRONICA
  ) |>
  drop_na()

set.seed(2026)

indice_entrenamiento <- sample(
  seq_len(nrow(covid_modelo)),
  size = floor(0.80 * nrow(covid_modelo))
)

covid_train <- covid_modelo[indice_entrenamiento, ]
covid_test <- covid_modelo[-indice_entrenamiento, ]

modelo_covid <- glm(
  MURIO ~ EDAD + NEUMONIA + DIABETES +
    HIPERTENSION + OBESIDAD + RENAL_CRONICA,
  data = covid_train,
  family = binomial()
)

covid_test$probabilidad <- predict(
  modelo_covid,
  newdata = covid_test,
  type = "response"
)

umbral <- 0.50

covid_test$prediccion <- ifelse(
  covid_test$probabilidad >= umbral,
  1,
  0
)

VP <- sum(
  covid_test$prediccion == 1 &
    covid_test$MURIO == 1
)

VN <- sum(
  covid_test$prediccion == 0 &
    covid_test$MURIO == 0
)

FP <- sum(
  covid_test$prediccion == 1 &
    covid_test$MURIO == 0
)

FN <- sum(
  covid_test$prediccion == 0 &
    covid_test$MURIO == 1
)

matriz_covid <- matrix(
  c(VN, FP, FN, VP),
  nrow = 2,
  byrow = TRUE,
  dimnames = list(
    Real = c("Clase 0", "Clase 1"),
    Predicha = c("Clase 0", "Clase 1")
  )
)

matriz_covid


### Métricas


In [ ]:
exactitud <- (VP + VN) / (VP + VN + FP + FN)
sensibilidad <- VP / (VP + FN)
especificidad <- VN / (VN + FP)
precision <- VP / (VP + FP)
f1 <- 2 * precision * sensibilidad /
  (precision + sensibilidad)
balanced_accuracy <- (
  sensibilidad + especificidad
) / 2

data.frame(
  exactitud,
  sensibilidad,
  especificidad,
  precision,
  f1,
  balanced_accuracy
)


### Comparación de umbrales


In [ ]:
evaluar_umbral <- function(umbral) {
  pred <- ifelse(
    covid_test$probabilidad >= umbral,
    1,
    0
  )

  vp <- sum(pred == 1 & covid_test$MURIO == 1)
  vn <- sum(pred == 0 & covid_test$MURIO == 0)
  fp <- sum(pred == 1 & covid_test$MURIO == 0)
  fn <- sum(pred == 0 & covid_test$MURIO == 1)

  data.frame(
    umbral = umbral,
    sensibilidad = vp / (vp + fn),
    especificidad = vn / (vn + fp),
    precision = vp / (vp + fp)
  )
}

resultados_umbrales <- do.call(
  rbind,
  lapply(
    seq(0.10, 0.90, by = 0.05),
    evaluar_umbral
  )
)

resultados_umbrales


En un problema de detección de riesgo, disminuir el umbral suele aumentar la
sensibilidad, pero también puede generar más falsos positivos. El umbral no es
una constante universal: depende del objetivo y del costo de cada error.

## Comparación integrada de modelos con COVID-19

Los capítulos anteriores aplicaron distintos algoritmos a COVID-19. Para compararlos de manera más coherente construiremos ahora un **benchmark didáctico común**: todos los modelos usarán las mismas variables, la misma muestra, la misma partición de entrenamiento/prueba y las mismas métricas.

> **Uso académico:** esta comparación sirve para estudiar diferencias entre algoritmos. No es una evaluación clínica ni debe utilizarse para pronóstico individual.

### Una muestra balanceada para comparar algoritmos

La variable `MURIO` está muy desbalanceada en la base original. Si evaluáramos únicamente exactitud sobre una muestra aleatoria, un modelo podría obtener un valor alto simplemente prediciendo casi siempre la clase mayoritaria.

Para este ejercicio tomamos, de forma reproducible, hasta 2 500 observaciones de cada clase. Esto produce un banco de pruebas balanceado. **Las métricas obtenidas describen este benchmark educativo y no la prevalencia de defunción en la población.**


In [ ]:
ruta_covid <- "datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz"

if (file.exists(ruta_covid)) {
  covid_comp_base <- readr::read_csv(ruta_covid, show_col_types = FALSE) |>
    dplyr::select(
      MURIO, EDAD, NEUMONIA, DIABETES, HIPERTENSION,
      OBESIDAD, RENAL_CRONICA, NUM_COMORBILIDADES
    ) |>
    tidyr::drop_na() |>
    dplyr::mutate(
      MURIO = factor(
        MURIO,
        levels = c(0, 1),
        labels = c("Sin defunción", "Defunción")
      )
    )

  set.seed(2026)
  n_por_clase <- min(
    2500,
    min(table(covid_comp_base$MURIO))
  )

  covid_comp <- covid_comp_base |>
    dplyr::group_by(MURIO) |>
    dplyr::slice_sample(n = n_por_clase) |>
    dplyr::ungroup()

  set.seed(2026)
  idx_train <- unlist(lapply(
    split(seq_len(nrow(covid_comp)), covid_comp$MURIO),
    function(i) sample(i, floor(0.80 * length(i)))
  ))

  train_comp <- covid_comp[idx_train, ]
  test_comp <- covid_comp[-idx_train, ]

  prop.table(table(train_comp$MURIO))
  prop.table(table(test_comp$MURIO))
}


### Función común de métricas


In [ ]:
metricas_covid_comunes <- function(real, predicho) {
  niveles <- c("Sin defunción", "Defunción")
  real <- factor(real, levels = niveles)
  predicho <- factor(predicho, levels = niveles)
  m <- table(Real = real, Predicho = predicho)

  VP <- m["Defunción", "Defunción"]
  FN <- m["Defunción", "Sin defunción"]
  FP <- m["Sin defunción", "Defunción"]
  VN <- m["Sin defunción", "Sin defunción"]

  div <- function(a, b) {
    if (is.na(b) || b == 0) return(NA_real_)
    as.numeric(a / b)
  }

  exactitud <- div(VP + VN, sum(m))
  sensibilidad <- div(VP, VP + FN)
  especificidad <- div(VN, VN + FP)
  precision <- div(VP, VP + FP)
  f1 <- if (is.na(precision) || is.na(sensibilidad) ||
            precision + sensibilidad == 0) {
    NA_real_
  } else {
    2 * precision * sensibilidad / (precision + sensibilidad)
  }

  data.frame(
    exactitud = exactitud,
    sensibilidad = sensibilidad,
    especificidad = especificidad,
    precision = precision,
    f1 = f1
  )
}


### 1. Regresión logística


In [ ]:
if (exists("train_comp")) {
  modelo_log_comp <- glm(
    MURIO ~ EDAD + NEUMONIA + DIABETES + HIPERTENSION +
      OBESIDAD + RENAL_CRONICA + NUM_COMORBILIDADES,
    data = train_comp,
    family = binomial
  )

  prob_log_comp <- predict(modelo_log_comp, test_comp, type = "response")
  pred_log_comp <- factor(
    ifelse(prob_log_comp >= 0.50, "Defunción", "Sin defunción"),
    levels = levels(train_comp$MURIO)
  )
}


### 2. k-NN


In [ ]:
if (exists("train_comp")) {
  vars_comp <- c(
    "EDAD", "NEUMONIA", "DIABETES", "HIPERTENSION",
    "OBESIDAD", "RENAL_CRONICA", "NUM_COMORBILIDADES"
  )

  X_train <- as.matrix(train_comp[vars_comp])
  X_test <- as.matrix(test_comp[vars_comp])

  medias <- apply(X_train, 2, mean)
  desv <- apply(X_train, 2, sd)
  desv[desv == 0] <- 1

  X_train_z <- scale(X_train, center = medias, scale = desv)
  X_test_z <- scale(X_test, center = medias, scale = desv)

  pred_knn_comp <- class::knn(
    train = X_train_z,
    test = X_test_z,
    cl = train_comp$MURIO,
    k = 11
  )
}


### 3. Árbol de decisión


In [ ]:
if (exists("train_comp")) {
  modelo_arbol_comp <- rpart::rpart(
    MURIO ~ EDAD + NEUMONIA + DIABETES + HIPERTENSION +
      OBESIDAD + RENAL_CRONICA + NUM_COMORBILIDADES,
    data = train_comp,
    method = "class",
    control = rpart::rpart.control(
      cp = 0.002,
      minsplit = 60,
      minbucket = 20,
      maxdepth = 5
    )
  )

  pred_arbol_comp <- predict(
    modelo_arbol_comp,
    test_comp,
    type = "class"
  )
}


### 4. Random Forest


In [ ]:
if (exists("train_comp")) {
  set.seed(2026)
  modelo_rf_comp <- ranger::ranger(
    MURIO ~ EDAD + NEUMONIA + DIABETES + HIPERTENSION +
      OBESIDAD + RENAL_CRONICA + NUM_COMORBILIDADES,
    data = train_comp,
    num.trees = 300,
    mtry = 3,
    min.node.size = 20,
    classification = TRUE,
    probability = FALSE,
    seed = 2026
  )

  pred_rf_comp <- predict(modelo_rf_comp, test_comp)$predictions
}


### 5. SVM radial


In [ ]:
if (exists("train_comp")) {
  set.seed(2026)
  modelo_svm_comp <- e1071::svm(
    MURIO ~ EDAD + NEUMONIA + DIABETES + HIPERTENSION +
      OBESIDAD + RENAL_CRONICA + NUM_COMORBILIDADES,
    data = train_comp,
    kernel = "radial",
    cost = 1,
    gamma = 1 / 7,
    scale = TRUE
  )

  pred_svm_comp <- predict(modelo_svm_comp, test_comp)
}


### 6. Naive Bayes


In [ ]:
if (exists("train_comp")) {
  train_nb_comp <- train_comp |>
    dplyr::mutate(
      NEUMONIA = factor(NEUMONIA),
      DIABETES = factor(DIABETES),
      HIPERTENSION = factor(HIPERTENSION),
      OBESIDAD = factor(OBESIDAD),
      RENAL_CRONICA = factor(RENAL_CRONICA)
    )

  test_nb_comp <- test_comp |>
    dplyr::mutate(
      NEUMONIA = factor(NEUMONIA, levels = levels(train_nb_comp$NEUMONIA)),
      DIABETES = factor(DIABETES, levels = levels(train_nb_comp$DIABETES)),
      HIPERTENSION = factor(HIPERTENSION, levels = levels(train_nb_comp$HIPERTENSION)),
      OBESIDAD = factor(OBESIDAD, levels = levels(train_nb_comp$OBESIDAD)),
      RENAL_CRONICA = factor(RENAL_CRONICA, levels = levels(train_nb_comp$RENAL_CRONICA))
    )

  modelo_nb_comp <- e1071::naiveBayes(
    MURIO ~ EDAD + NEUMONIA + DIABETES + HIPERTENSION +
      OBESIDAD + RENAL_CRONICA + NUM_COMORBILIDADES,
    data = train_nb_comp,
    laplace = 1
  )

  pred_nb_comp <- predict(modelo_nb_comp, test_nb_comp, type = "class")
}


### 7. Red neuronal


In [ ]:
if (exists("train_comp")) {
  train_nn_comp <- as.data.frame(X_train_z)
  test_nn_comp <- as.data.frame(X_test_z)
  train_nn_comp$MURIO_NUM <- ifelse(train_comp$MURIO == "Defunción", 1, 0)

  set.seed(2026)
  modelo_nn_comp <- neuralnet::neuralnet(
    MURIO_NUM ~ EDAD + NEUMONIA + DIABETES + HIPERTENSION +
      OBESIDAD + RENAL_CRONICA + NUM_COMORBILIDADES,
    data = train_nn_comp,
    hidden = 5,
    linear.output = FALSE,
    lifesign = "none",
    stepmax = 1e6
  )

  prob_nn_comp <- as.numeric(
    neuralnet::compute(
      modelo_nn_comp,
      test_nn_comp[vars_comp]
    )$net.result[, 1]
  )

  pred_nn_comp <- factor(
    ifelse(prob_nn_comp >= 0.50, "Defunción", "Sin defunción"),
    levels = levels(train_comp$MURIO)
  )
}


### Tabla comparativa final


In [ ]:
if (exists("pred_nn_comp")) {
  comparacion_covid_modelos <- dplyr::bind_rows(
    cbind(modelo = "Regresión logística", metricas_covid_comunes(test_comp$MURIO, pred_log_comp)),
    cbind(modelo = "k-NN (k=11)", metricas_covid_comunes(test_comp$MURIO, pred_knn_comp)),
    cbind(modelo = "Árbol de decisión", metricas_covid_comunes(test_comp$MURIO, pred_arbol_comp)),
    cbind(modelo = "Random Forest", metricas_covid_comunes(test_comp$MURIO, pred_rf_comp)),
    cbind(modelo = "SVM radial", metricas_covid_comunes(test_comp$MURIO, pred_svm_comp)),
    cbind(modelo = "Naive Bayes", metricas_covid_comunes(test_comp$MURIO, pred_nb_comp)),
    cbind(modelo = "Red neuronal", metricas_covid_comunes(test_comp$MURIO, pred_nn_comp))
  )

  comparacion_covid_modelos |>
    dplyr::mutate(
      dplyr::across(
        c(exactitud, sensibilidad, especificidad, precision, f1),
        ~ round(.x, 3)
      )
    )
}


In [ ]:
if (exists("comparacion_covid_modelos")) {
  comp_larga <- comparacion_covid_modelos |>
    tidyr::pivot_longer(
      cols = c(exactitud, sensibilidad, especificidad, precision, f1),
      names_to = "metrica",
      values_to = "valor"
    )

  ggplot2::ggplot(
    comp_larga,
    ggplot2::aes(x = modelo, y = valor, fill = metrica)
  ) +
    ggplot2::geom_col(position = "dodge") +
    ggplot2::coord_flip() +
    ggplot2::scale_y_continuous(limits = c(0, 1)) +
    ggplot2::labs(
      title = "Comparación común de modelos con COVID-19",
      subtitle = "Misma muestra balanceada, mismas variables y misma partición 80/20",
      x = NULL,
      y = "Valor de la métrica",
      fill = "Métrica"
    ) +
    tema_libro()
}


El modelo con mayor exactitud no necesariamente es el mejor para todos los objetivos. La sensibilidad prioriza detectar defunciones; la especificidad prioriza reconocer correctamente los casos sin defunción; precisión y F1 ofrecen otras perspectivas. Además, aquí se usaron hiperparámetros didácticos fijos, no una búsqueda exhaustiva de optimización.

### Lectura recomendada de la tabla

Al comparar los modelos conviene preguntar:

1. ¿Cuál ofrece el mejor equilibrio entre sensibilidad y especificidad?
2. ¿Algún modelo gana en exactitud pero pierde mucha sensibilidad?
3. ¿La mayor complejidad de Random Forest, SVM o la red neuronal produce una mejora suficiente frente a la regresión logística?
4. ¿Un modelo sencillo sería preferible si ofrece resultados parecidos y mayor interpretabilidad?
5. ¿Cambiaría la conclusión si la muestra conservara la fuerte desproporción original entre clases?

Esta comparación cierra la ruta supervisada del caso COVID-19 y conecta los capítulos de algoritmos con una evaluación común y reproducible.
